# Setup

Here we will install the seegnature library to have access to the models and the different functions to evaluate them.

In [1]:
!wget https://anonymous.4open.science/api/repo/seegnature_ECML26-272A/zip -O main.zip
!unzip main.zip -d /content/seegnature_ECML26

--2026-03-20 09:52:55--  https://anonymous.4open.science/api/repo/seegnature_ECML26-272A/zip
Resolving anonymous.4open.science (anonymous.4open.science)... 172.67.183.76, 104.21.18.195, 2606:4700:3035::ac43:b74c, ...
Connecting to anonymous.4open.science (anonymous.4open.science)|172.67.183.76|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘main.zip’

main.zip                [    <=>             ] 358.65M   338KB/s    in 19m 41s 

2026-03-20 10:12:39 (311 KB/s) - ‘main.zip’ saved [376071526]

Archive:  main.zip
  inflating: /content/seegnature_ECML26/.gitignore  
  inflating: /content/seegnature_ECML26/LICENSE  
  inflating: /content/seegnature_ECML26/README.md  
  inflating: /content/seegnature_ECML26/data/mne/sub-00000-epo.fif  
  inflating: /content/seegnature_ECML26/data/mne/sub-27670-epo.fif  
  inflating: /content/seegnature_ECML26/data/mne/sub-27817-epo.fif  
  inflating: /content/seegnature_ECML26/data/mne/sub-

In [2]:
!pip install pysiglib
!pip install braindecode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.1/288.1 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pysiglib: filename=pysiglib-2.0.1-py3-none-any.whl size=408992 sha256=039f526fd6eae7980b075dc97e87c3b6fecb5e7800397d9fc1574499c8d1d388
  Stored in directory: /root/.cache/pip/wheels/56/53/c2/4b00573b820651ec91f5a32e01982c3b24322fd8b7217f2ea3
Successfully built pysiglib
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 448.8/448.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.4/178.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.5/268.5 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 146.4 MB/s eta 0:00:00
  Attempting uninstall: pa

We will also download the experiment data to use it for the comparison.

Here are the necessary imports:


In [3]:
import sys
sys.path.append("/content/seegnature_ECML26/src/")
!pip install -e /content/seegnature_ECML26

Obtaining file:///content/seegnature_ECML26
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 172.6 MB/s eta 0:00:00
  Building editable for seegnature (pyproject.toml) ... done
  Created wheel for seegnature: filename=seegnature-0.1.0-0.editable-py3-none-any.whl size=2569 sha256=94c4f60fcbdb23ae65cae866d4f436a343de2ab027a87412bafbf515d4c720bd
  Stored in directory: /tmp/pip-ephem-wheel-cache-kzs5rgp1/wheels/d7/03/d2/781849278f09e21736d67ec3ade6c6696cd0e14adf9c4ce62c
Successfully built seegnature
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found 

In [4]:
import matplotlib.pyplot as plt

from seegnature.evaluation import evaluate_models_with_subject_dict
from seegnature.models import SigLR
from seegnature.models.concurrents import (

    ATCNetClassifier,
    CSPLDA,
    CSPSVM,
    EEGConformerClassifier,
    EEGNetClassifier
)
from seegnature.utils import subject_dict_from_files

Here are the variables controlling this notebook's behaviour:

In [5]:
SUBJECT_FILES = [
    "seegnature_ECML26/data/mne/sub-00000-epo.fif",
    "seegnature_ECML26/data/mne/sub-27670-epo.fif",
    "seegnature_ECML26/data/mne/sub-27817-epo.fif",
    "seegnature_ECML26/data/mne/sub-27863-epo.fif"
    ]
SEED=42
DEGREE=2

# Data loading

In this part we will load the data. We have multiple participants with a EEG recording each. We will load the data following a mapping format between the participant and a tuple with the EEG recording and the label to be predicted by a classifier.

In [6]:

subject_dict = subject_dict_from_files(SUBJECT_FILES)
print(f"Loaded subjects are {", ".join(subject_dict.keys())}")

Reading /content/seegnature_ECML26/data/mne/sub-00000-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    3898.44 ms
        0 CTF compensation matrices available
Adding metadata with 1 columns
256 matching events found
No baseline correction applied
0 projection items activated
Reading /content/seegnature_ECML26/data/mne/sub-27670-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    3898.44 ms
        0 CTF compensation matrices available
Adding metadata with 1 columns
242 matching events found
No baseline correction applied
0 projection items activated
Reading /content/seegnature_ECML26/data/mne/sub-27817-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    3898.44 ms
        0 CTF compensation matrices available
Adding metadata with 1 columns
300 matching events found
No baseline correction applied
0 projection items activated
Reading /content/seegnature_ECML26/data/mne/sub-27863-epo.fif ...
    Found the data of intere

The data follows a "subject : (EEG, label)" format. The EEG itself is an array of shape (batch, channel, time). The label is an array of strings, here either "control_memoranda" or "encoding_memoranda".

The whole epoch is present, we only want the mental imagery part of it which starts at index -375 on the time axis. Similarly, to have access to additional scores for the models such as the F1-score or the AUC, we should get rid of the strings and transform them into either 0 or 1. This allows us to have a binary classification problem.

In [7]:
subject_dict = {subject: (X[..., -375:], (y == "encoding_memoranda").astype(int)) for subject, (X, y) in subject_dict.items()}

# Normalization comparison

Now that our data is prepared, we can use it to evaluate models using `evaluate_models_with_subject_dict`. The function takes a mapping of model names to their respective Python object. It returns a DataFrame from the `pandas` package.

Here we will experiment with our model `SigLR` to see if normalization as an impact on the scores. By normalization, we mean here a substraction of the mean value of the EEG along the time axis and a division by its maximum amplitude.

We just have to create the model in a dictionary and a cross validation with 5 folds will be done for each of them for each subject.

In [8]:
norm_models = {
    "siglr_not_normalized": SigLR(DEGREE, normalization=False, random_state=SEED),
    "siglr_normalized": SigLR(DEGREE, normalization=True, random_state=SEED),
}

In [ ]:
norm_df = evaluate_models_with_subject_dict(norm_models, subject_dict, random_state=SEED)
norm_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    2.3s remaining:    3.5s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    2.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    2.1s remaining:    3.1s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    2.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    2.1s remaining:    3.1s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    2.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    2.1s remaining:    3.1s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    2.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.
[Parall

train                      \
                                     accuracy        f1   roc_auc   
model                subject   fold                                 
siglr_not_normalized sub-00000 0     0.509804  0.675325  0.619231   
                               1     0.512195  0.677419  0.599905   
                               2     0.512195  0.677419  0.618286   
                               3     0.512195  0.677419  0.633810   
                               4     0.512195  0.677419  0.643333   
                     sub-27670 0     0.502591  0.668966  0.682882   
                               1     0.502591  0.668966  0.693406   
                               2     0.505155  0.671233  0.737457   
                               3     0.505155  0.671233  0.675595   
                               4     0.505155  0.671233  0.695472   
                     sub-27817 0     0.500000  0.000000  0.500000   
                               1     0.500000  0.000000  0.500000   
                               2     0.500000  0.000000  0.500000   
                               3     0.500000  0.000000  0.500000   
                               4     0.500000  0.000000  0.500000   
                     sub-27863 0     0.552239  0.711538  0.733934   
                               1     0.552239  0.711538  0.713514   
                               2     0.549505  0.709265  0.744283   
                               3     0.549505  0.709265  0.743788   
                               4     0.554455  0.713376  0.719345   
siglr_normalized     sub-00000 0     1.000000  1.000000  1.000000   
                               1     1.000000  1.000000  1.000000   
                               2     1.000000  1.000000  1.000000   
                               3     1.000000  1.000000  1.000000   
                               4     1.000000  1.000000  1.000000   
                     sub-27670 0     1.000000  1.000000  1.000000   
                               1     1.000000  1.000000  1.000000   
                               2     1.000000  1.000000  1.000000   
                               3     1.000000  1.000000  1.000000   
                               4     1.000000  1.000000  1.000000   
                     sub-27817 0     1.000000  1.000000  1.000000   
                               1     1.000000  1.000000  1.000000   
                               2     1.000000  1.000000  1.000000   
                               3     1.000000  1.000000  1.000000   
                               4     1.000000  1.000000  1.000000   
                     sub-27863 0     1.000000  1.000000  1.000000   
                               1     1.000000  1.000000  1.000000   
                               2     1.000000  1.000000  1.000000   
                               3     1.000000  1.000000  1.000000   
                               4     1.000000  1.000000  1.000000   

                                                        test            \
                                    d2_brier_score  accuracy        f1   
model                subject   fold                                      
siglr_not_normalized sub-00000 0     -9.068226e-09  0.519231  0.683544   
                               1     -1.397413e-08  0.509804  0.675325   
                               2     -1.396215e-08  0.509804  0.675325   
                               3     -1.395017e-08  0.509804  0.675325   
                               4     -1.395331e-08  0.509804  0.675325   
                     sub-27670 0     -2.058176e-11  0.510204  0.675676   
                               1      1.672658e-10  0.510204  0.675676   
                               2     -1.499064e-09  0.500000  0.666667   
                               3     -2.101772e-09  0.500000  0.666667   
                               4     -2.019946e-09  0.500000  0.666667   
                     sub-27817 0      0.000000e+00  0.500000  0.000000   
                               1    

The returned dataframe has multiple index levels allowing us to group results by models, subjects and even folds. In terms of columns, we have three main column: train, test and seed. train and test are both composite columns with the scores.

We can now group our rows by model and see the average scores and their standard deviation.

In [ ]:
norm_agg = norm_df["test"].groupby(level=0).agg(["mean", "std"]).stack(0).swaplevel(0,1).sort_index()
norm_agg

mean       std
               model                                   
accuracy       siglr_normalized      0.774076  0.042926
               siglr_not_normalized  0.516845  0.021635
d2_brier_score siglr_normalized      0.226502  0.125161
               siglr_not_normalized -0.000152  0.000220
f1             siglr_normalized      0.779314  0.046412
               siglr_not_normalized  0.514556  0.305238
roc_auc        siglr_normalized      0.846929  0.036220
               siglr_not_normalized  0.614199  0.116326

Those results show us that normalization has a substantial impact on the model's training. Consequently, the normalization will be activated by default from this point onward.

# Augmentation comparison

We have used two path augmentations in our computation of the signature: the time augmentation and the lead-lag augmentation. We will see how they impact the results of the model.

In [ ]:
aug_models = {
    "siglr_no_aug": SigLR(DEGREE, time_aug=False, lead_lag_aug=False, random_state=SEED),
    "siglr_time_aug": SigLR(DEGREE, time_aug=True, lead_lag_aug=False, random_state=SEED),
    "siglr_lead_lag_aug": SigLR(DEGREE, time_aug=False, lead_lag_aug=True, random_state=SEED),
    "siglr_both_aug": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, random_state=SEED),
}

In [ ]:
aug_df = evaluate_models_with_subject_dict(aug_models, subject_dict, random_state=SEED)
aug_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    1.7s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    1.6s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.4s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out

train                                  test  \
                              accuracy   f1 roc_auc d2_brier_score  accuracy   
model          subject   fold                                                  
siglr_no_aug   sub-00000 0         1.0  1.0     1.0       0.999932  0.807692   
                         1         1.0  1.0     1.0       0.999927  0.764706   
                         2         1.0  1.0     1.0       0.999924  0.745098   
                         3         1.0  1.0     1.0       0.999904  0.764706   
                         4         1.0  1.0     1.0       0.999898  0.764706   
...                                ...  ...     ...            ...       ...   
siglr_both_aug sub-27863 0         1.0  1.0     1.0       0.999999  0.764706   
                         1         1.0  1.0     1.0       1.000000  0.803922   
                         2         1.0  1.0     1.0       1.000000  0.860000   
                         3         1.0  1.0     1.0       0.999999  0.860000   
                         4         1.0  1.0     1.0       0.999999  0.880000   

                                                                 seed  
                                     f1   roc_auc d2_brier_score       
model          subject   fold                                          
siglr_no_aug   sub-00000 0     0.807692  0.822222       0.245304   42  
                         1     0.750000  0.840000       0.145091   42  
                         2     0.754717  0.838462       0.189717   42  
                         3     0.800000  0.886154       0.156553   42  
                         4     0.760000  0.898462       0.298901   42  
...                                 ...       ...            ...  ...  
siglr_both_aug sub-27863 0     0.806452  0.857143       0.187678   42  
                         1     0.833333  0.830745       0.190949   42  
                         2     0.862745  0.913961       0.466379   42  
                         3     0.877193  0.896104       0.468049   42  
                         4     0.892857  0.924316       0.577215   42  

[80 rows x 9 columns]

In [ ]:

aug_agg = aug_df["test"].groupby(level=0).agg(["mean", "std"]).stack(0).swaplevel(0,1).sort_index()
aug_agg

mean       std
               model                                 
accuracy       siglr_both_aug      0.843682  0.059676
               siglr_lead_lag_aug  0.843704  0.061335
               siglr_no_aug        0.774076  0.042926
               siglr_time_aug      0.777952  0.042927
d2_brier_score siglr_both_aug      0.436062  0.227677
               siglr_lead_lag_aug  0.437852  0.224816
               siglr_no_aug        0.226502  0.125161
               siglr_time_aug      0.238849  0.124154
f1             siglr_both_aug      0.851476  0.054859
               siglr_lead_lag_aug  0.851979  0.056023
               siglr_no_aug        0.779314  0.046412
               siglr_time_aug      0.783153  0.046344
roc_auc        siglr_both_aug      0.898240  0.054924
               siglr_lead_lag_aug  0.897267  0.056200
               siglr_no_aug        0.846929  0.036220
               siglr_time_aug      0.852548  0.035407

Those results show that the lead lag augmentation has an impact on the model's performances. Time augmentation in contrast enhances the efficiency but to a lower degree. We will apply both augmentations from this point onward.

# Penalty comparison

The logistic regression part of the model has a penalty on its parameters. This penalty can be $\ell_1$ or $\ell_2$. Up until that point the model was tested using a regular $\ell_2$ penalty. Here we will try to determine the impact of the penalty's choice.

In [ ]:
DEGREE = 2
penalty_models = {
    "siglr_l1": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, l1_ratio=1.0, random_state=SEED),
    "siglr_l2": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, l1_ratio=0.0, random_state=SEED),
}

In [ ]:
penalty_df = evaluate_models_with_subject_dict(penalty_models, subject_dict, random_state=SEED)
penalty_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    9.6s remaining:   14.5s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    9.7s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   10.1s remaining:   15.2s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   10.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   11.3s remaining:   17.0s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   11.5s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   10.0s remaining:   15.0s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   10.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.
[Parall

train                                     \
                         accuracy        f1   roc_auc d2_brier_score   
model    subject   fold                                                
siglr_l1 sub-00000 0     0.965686  0.967136  0.997788       0.862763   
                   1     0.960976  0.962617  0.993619       0.835253   
                   2     0.956098  0.958140  0.994095       0.840446   
                   3     0.960976  0.962617  0.995524       0.818113   
                   4     0.956098  0.957746  0.991143       0.815475   
         sub-27670 0     1.000000  1.000000  1.000000       0.951255   
                   1     1.000000  1.000000  1.000000       0.944812   
                   2     1.000000  1.000000  1.000000       0.946350   
                   3     1.000000  1.000000  1.000000       0.950853   
                   4     1.000000  1.000000  1.000000       0.939786   
         sub-27817 0     1.000000  1.000000  1.000000       0.979279   
                   1     1.000000  1.000000  1.000000       0.981200   
                   2     1.000000  1.000000  1.000000       0.980669   
                   3     1.000000  1.000000  1.000000       0.982097   
                   4     1.000000  1.000000  1.000000       0.980741   
         sub-27863 0     0.995025  0.995475  0.998999       0.935296   
                   1     0.990050  0.990991  0.999199       0.943649   
                   2     1.000000  1.000000  1.000000       0.947188   
                   3     0.995050  0.995475  0.999802       0.937044   
                   4     0.995050  0.995516  0.999702       0.947497   
siglr_l2 sub-00000 0     1.000000  1.000000  1.000000       0.999999   
                   1     1.000000  1.000000  1.000000       0.999999   
                   2     1.000000  1.000000  1.000000       0.999999   
                   3     1.000000  1.000000  1.000000       0.999999   
                   4     1.000000  1.000000  1.000000       0.999999   
         sub-27670 0     1.000000  1.000000  1.000000       0.999997   
                   1     1.000000  1.000000  1.000000       0.999997   
                   2     1.000000  1.000000  1.000000       0.999997   
                   3     1.000000  1.000000  1.000000       0.999997   
                   4     1.000000  1.000000  1.000000       0.999997   
         sub-27817 0     1.000000  1.000000  1.000000       0.999998   
                   1     1.000000  1.000000  1.000000       0.999999   
                   2     1.000000  1.000000  1.000000       0.999998   
                   3     1.000000  1.000000  1.000000       0.999999   
                   4     1.000000  1.000000  1.000000       0.999998   
         sub-27863 0     1.000000  1.000000  1.000000       0.999999   
                   1     1.000000  1.000000  1.000000       1.000000   
                   2     1.000000  1.000000  1.000000       1.000000   
                   3     1.000000  1.000000  1.000000       0.999999   
                   4     1.000000  1.000000  1.000000       0.999999   

                             test                                    seed  
                         accuracy        f1   roc_auc d2_brier_score       
model    subject   fold                                                    
siglr_l1 sub-00000 0     0.788462  0.807018  0.777778       0.250311   42  
                   1     0.843137  0.851852  0.855385       0.445167   42  
                   2     0.823529  0.830189  0.841538       0.371201   42  
                   3     0.823529  0.842105  0.909231       0.536306   42  
                   4     0.882353  0.888889  0.930769       0.621946   42  
         sub-27670 0     0.755102  0.750000  0.870000       0.410604   42  
                   1     0.795918  0.807692  0.858333       0.397396   42  
                   2     0.854167  0.851064  0.918403       0.521658   42  
                   3     0.812500  0.823529  0.885417       0.421447   42  
                   

In [ ]:
penalty_df["test"]["accuracy"]["siglr_l2"].groupby(level=0).agg(["mean", "std"])

,mean,std
subject,,
sub-00000,0.797059,0.045932
sub-27670,0.830612,0.026447
sub-27817,0.913333,0.051908
sub-27863,0.833725,0.047891


In [ ]:
penalty_agg = penalty_df["test"].groupby(level=0).agg(["mean", "std"]).stack(0).swaplevel(0,1).sort_index()
penalty_agg

mean       std
               model                       
accuracy       siglr_l1  0.850055  0.058543
               siglr_l2  0.843682  0.059676
d2_brier_score siglr_l1  0.539756  0.171124
               siglr_l2  0.436062  0.227677
f1             siglr_l1  0.858271  0.054634
               siglr_l2  0.851476  0.054859
roc_auc        siglr_l1  0.904679  0.052978
               siglr_l2  0.898240  0.054924

# 3rd order of signature

In [ ]:
DEGREE=3
penalty_models = {
    #"siglr_l1": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, l1_ratio=1.0, random_state=SEED),
    "siglr_l2": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, l1_ratio=0.0, random_state=SEED),
}
penalty_df = evaluate_models_with_subject_dict(penalty_models, subject_dict, random_state=SEED)
penalty_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 26 concurrent workers.


KeyboardInterrupt: 

In [ ]:
penalty_agg = penalty_df["test"].groupby(level=0).agg(["mean", "std"]).stack(0).swaplevel(0,1).sort_index()
penalty_agg

Here, we can see that the $\ell_1$ penalty enhances the performances of the model. It is also especially useful for the extraction of the graphs. However, the gain in performance is heavily skewed by the augmented time of execution. For this reason, the $\ell_2$ penalty will be used from this point onward.

# Literature comparison

The good scores obtained by the model beg the question of how they compare to the ones obtained by other well known models of the literature. We wanted to compare the model with different approaches to the task.

For a more traditional approach to EEG analysis, we have settled on a Common Spatial Patterns (CSP) feature extraction of the signal and either a Linear Discriminant Analysis (LDA) or a State Vector Machine (SVM) for the classification. This choice follows the review by Xie and Oniga [TODO] and is backed by its usage in the comparative study of Aljalal and Djemal [TODO]. The tools used come from the MNE library [TODO] and SciKit [TODO].

For a deep-learning convolution-based approach, we have chosen EEGNet by Lawhern et al. [TODO]. It is usable on a variety of EEG analysis disciplines and its usage is widespread in the literature. The implementation used comes from the Baindecode toolbox [TODO] which also provides the last two models.

For a deep-learning attention-based approach, we have chosen EEG Conformer by Sing et al. [TODO] and ATCNet by Altaheri et al. [TODO]. They are general-purpose models which use attention and tokenization of the EEG signal. They also obtain good performances and are good candidates for this experiment.

The model configuration we choose to fare up against other models of the literature is the normalization + time and lead lag augmentations + $\ell_2$ penalty one.

**CAUTION:** *We have modified CSP_SVM to include Brier Score and AUC recovery*.

In [12]:
literature_models = {
    "siglr_l1": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, random_state=SEED, l1_ratio = 1.0),
    "siglr_l2": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, random_state=SEED),
    "csp_lda": CSPLDA(),
    "csp_svm": CSPSVM(random_state=SEED),
    "eegnet": EEGNetClassifier(data_freq=128, random_state=SEED),
    "eeg_conformer": EEGConformerClassifier(data_freq=128, random_state=SEED),
    "atcnet": ATCNetClassifier(data_freq=128, random_state=SEED)
}

Most of the hyperparameters have been gathered from the different papers and example on the documentations.

In [13]:
literature_df = evaluate_models_with_subject_dict(literature_models, subject_dict, random_state=SEED)
literature_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    7.5s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    7.0s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    7.9s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    7.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    2.9s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    2.8s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out

train                                     \
                         accuracy        f1   roc_auc d2_brier_score   
model    subject   fold                                                
siglr_l1 sub-00000 0     0.965686  0.967136  0.997788       0.862763   
                   1     0.960976  0.962617  0.993619       0.835253   
                   2     0.956098  0.958140  0.994095       0.840446   
                   3     0.960976  0.962617  0.995524       0.818113   
                   4     0.956098  0.957746  0.991143       0.815475   
...                           ...       ...       ...            ...   
atcnet   sub-27863 0     0.815920  0.856031  0.986236       0.439851   
                   1     0.955224  0.960699  0.994695       0.849026   
                   2     0.955446  0.960352  0.988070       0.843394   
                   3     0.628713  0.489796  0.981784      -0.244703   
                   4     0.554455  0.713376  0.982738      -0.782293   

                             test                                    seed  
                         accuracy        f1   roc_auc d2_brier_score       
model    subject   fold                                                    
siglr_l1 sub-00000 0     0.788462  0.807018  0.777778       0.250311   42  
                   1     0.843137  0.851852  0.855385       0.445167   42  
                   2     0.823529  0.830189  0.841538       0.371201   42  
                   3     0.823529  0.842105  0.909231       0.536306   42  
                   4     0.882353  0.888889  0.930769       0.621946   42  
...                           ...       ...       ...            ...  ...  
atcnet   sub-27863 0     0.666667  0.760563  0.799689      -0.181889   42  
                   1     0.764706  0.812500  0.857143       0.176669   42  
                   2     0.760000  0.812500  0.832792       0.135275   42  
                   3     0.520000  0.250000  0.863636      -0.705326   42  
                   4     0.540000  0.701299  0.882448      -0.844155   42  

[140 rows x 9 columns]

In [14]:
literature_agg = literature_df["test"].groupby(level=0).agg(["mean", "std"]).stack(0).swaplevel(0,1).sort_index()
literature_agg

mean       std
               model                            
accuracy       atcnet         0.634925  0.112140
               csp_lda        0.846745  0.057421
               csp_svm        0.796082  0.073085
               eeg_conformer  0.498060  0.027619
               eegnet         0.579644  0.089973
               siglr_l1       0.850055  0.058543
               siglr_l2       0.843682  0.059676
d2_brier_score atcnet        -0.292712  0.472928
               csp_lda        0.532208  0.166203
               csp_svm        0.439943  0.177310
               eeg_conformer -0.577251  0.384159
               eegnet        -0.558018  0.415016
               siglr_l1       0.539756  0.171124
               siglr_l2       0.436062  0.227677
f1             atcnet         0.565100  0.290429
               csp_lda        0.855837  0.053565
               csp_svm        0.812634  0.066935
               eeg_conformer  0.339789  0.348818
               eegnet         0.526515  0.287073
               siglr_l1       0.858271  0.054634
               siglr_l2       0.851476  0.054859
roc_auc        atcnet         0.782456  0.116777
               csp_lda        0.911982  0.056158
               csp_svm        0.880590  0.063610
               eeg_conformer  0.576488  0.114593
               eegnet         0.743957  0.131654
               siglr_l1       0.904679  0.052978
               siglr_l2       0.898240  0.054924

#Inter-participant study

We start by building the dataset

In [8]:
import numpy as np
import os
os.makedirs("seegnature_ECML26/data/raw_numpy/inter", exist_ok=True)
SUBJECT_FILES_numpy = [
    "seegnature_ECML26/data/raw_numpy/sub-00000/sub-00000_array-epochs.npy",
     "seegnature_ECML26/data/raw_numpy/sub-27670/sub-27670_array-epochs.npy",
     "seegnature_ECML26/data/raw_numpy/sub-27817/sub-27817_array-epochs.npy",
     "seegnature_ECML26/data/raw_numpy/sub-27863/sub-27863_array-epochs.npy"
    ]
SEED=42

inter = np.concatenate([np.load(file) for file in SUBJECT_FILES_numpy])

In [9]:
np.save("seegnature_ECML26/data/raw_numpy/inter/inter_array-epochs.npy", inter, allow_pickle = True)

In [10]:
y_FILES_numpy = [
    "seegnature_ECML26/data/raw_numpy/sub-00000/sub-00000-labels.npy",
     "seegnature_ECML26/data/raw_numpy/sub-27670/sub-27670-labels.npy",
     "seegnature_ECML26/data/raw_numpy/sub-27817/sub-27817-labels.npy",
     "seegnature_ECML26/data/raw_numpy/sub-27863/sub-27863-labels.npy"
    ]
inter_y = np.concatenate([np.load(file, allow_pickle=True) for file in y_FILES_numpy])

In [11]:
np.save("seegnature_ECML26/data/raw_numpy/inter/inter-labels.npy", inter_y, allow_pickle=True)

In [12]:
channel_coords = np.load("/content/seegnature_ECML26/data/raw_numpy/sub-00000/sub-00000-channel_coords.npy")

In [13]:
np.save("seegnature_ECML26/data/raw_numpy/inter/inter-channel_coords.npy", channel_coords)

In [14]:
channel_names = np.load("/content/seegnature_ECML26/data/raw_numpy/sub-00000/sub-00000-channel_names.npy")

In [15]:
np.save("seegnature_ECML26/data/raw_numpy/inter/inter-channel_names.npy", channel_names)

In [16]:
import glob
from pathlib import Path

import mne

from seegnature.utils import numpy_to_mne

# ========== VARIABLES ==========

RAW_NUMPY_DIR = "/content/seegnature_ECML26/data/raw_numpy"
MNE_DIR = "/content/seegnature_ECML26/data/mne"
SAMPLING_FREQ = 128

# ========== EXECUTION ==========

print(f"Looking through \033[36m{RAW_NUMPY_DIR}\033[0m for subject folders")

raw_numpy_dir_path = Path(RAW_NUMPY_DIR)
mne_dir_path = Path(MNE_DIR)

subject_folders = glob.glob(str(raw_numpy_dir_path / "*"))

for subject in subject_folders:
    subject = Path(subject)
    if not subject.is_dir():
        continue
    hypothetical_result = mne_dir_path / f"{subject.name}-epo.fif"
    print(
        f"Found subject folder \033[36m{subject.name}\033[0m, attempting conversion into \033[36m{hypothetical_result}\033[0m..."
    )
    try:
        epochs: mne.Epochs = numpy_to_mne(subject, SAMPLING_FREQ)
        epochs.save(hypothetical_result)
        print("---> \033[32mSucceeded\033[0m")
    except Exception as e:
        print(f"---> \033[31mFailed\033[0m ({e})")

Looking through /content/seegnature_ECML26/data/raw_numpy for subject folders
Found subject folder sub-27817, attempting conversion into /content/seegnature_ECML26/data/mne/sub-27817-epo.fif...
Adding metadata with 1 columns
300 matching events found
No baseline correction applied
0 projection items activated
---> Failed (Destination file exists. Please use option "overwrite=True" to force overwriting.)
Found subject folder sub-27670, attempting conversion into /content/seegnature_ECML26/data/mne/sub-27670-epo.fif...
Adding metadata with 1 columns
242 matching events found
No baseline correction applied
0 projection items activated
---> Failed (Destination file exists. Please use option "overwrite=True" to force overwriting.)
Found subject folder inter, attempting conversion into /content/seegnature_ECML26/data/mne/inter-epo.fif...
Adding metadata with 1 columns
1050 matching events found
No baseline correction applied
0 projection items activated
---> Succeeded
Found subject folder su

In [17]:
SUBJECT_FILES = [
    "seegnature_ECML26/data/mne/inter-epo.fif"
    ]

subject_dict = subject_dict_from_files(SUBJECT_FILES)
print(f"Loaded subjects are {", ".join(subject_dict.keys())}")
subject_dict = {subject: (X[..., -375:], (y == "encoding_memoranda").astype(int)) for subject, (X, y) in subject_dict.items()}
SEED=42

Reading /content/seegnature_ECML26/data/mne/inter-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    3898.44 ms
        0 CTF compensation matrices available
Adding metadata with 1 columns
1050 matching events found
No baseline correction applied
0 projection items activated
Loaded subjects are inter


## Literature and Sig2+$\ell_2$ comparison

In [18]:
literature_models = {
    "siglr": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, random_state=SEED),
    "csp_lda": CSPLDA(),
    "csp_svm": CSPSVM(random_state=SEED),
    "eegnet": EEGNetClassifier(data_freq=128, random_state=SEED),
    "eeg_conformer": EEGConformerClassifier(data_freq=128, random_state=SEED),
    "atcnet": ATCNetClassifier(data_freq=128, random_state=SEED)
}
literature_df = evaluate_models_with_subject_dict(literature_models, subject_dict, random_state=SEED)
literature_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    8.4s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    3.8s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    3.9s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  1.8min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  4.8min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  3.8min finished


train                                     \
                            accuracy        f1   roc_auc d2_brier_score   
model         subject fold                                                
siglr         inter   0     1.000000  1.000000  1.000000       0.999992   
                      1     1.000000  1.000000  1.000000       0.999993   
                      2     1.000000  1.000000  1.000000       0.999994   
                      3     1.000000  1.000000  1.000000       0.999994   
                      4     1.000000  1.000000  1.000000       0.999993   
csp_lda       inter   0     0.611905  0.635347  0.682570       0.112831   
                      1     0.652381  0.659674  0.705148       0.130876   
                      2     0.622619  0.643420  0.689394       0.115984   
                      3     0.617857  0.630610  0.683066       0.109778   
                      4     0.603571  0.629588  0.680212       0.112030   
csp_svm       inter   0     0.660714  0.734390  0.750265       0.177753   
                      1     0.698810  0.745729  0.780839       0.221373   
                      2     0.690476  0.743083  0.796667       0.255977   
                      3     0.677381  0.729811  0.776322       0.222846   
                      4     0.650000  0.724203  0.751459       0.176124   
eegnet        inter   0     0.561905  0.701783  0.954492      -0.691028   
                      1     0.529762  0.686757  0.640407      -0.877326   
                      2     0.904762  0.902676  0.962890       0.676053   
                      3     0.638095  0.740171  0.939391      -0.371165   
                      4     0.664286  0.751761  0.919014      -0.223375   
eeg_conformer inter   0     0.484524  0.000000  0.422840      -0.964850   
                      1     0.515476  0.680283  0.629824      -0.936439   
                      2     0.483333  0.000000  0.614895      -0.886985   
                      3     0.516667  0.681319  0.649534      -0.925215   
                      4     0.483333  0.000000  0.645987      -1.068504   
atcnet        inter   0     0.613095  0.726661  0.954440      -0.236136   
                      1     0.759524  0.809434  0.947138       0.219318   
                      2     0.588095  0.339695  0.964853      -0.342273   
                      3     0.629762  0.735769  0.966737      -0.218073   
                      4     0.633333  0.737649  0.963537      -0.337556   

                                test                                    seed  
                            accuracy        f1   roc_auc d2_brier_score       
model         subject fold                                                    
siglr         inter   0     0.790476  0.792453  0.874830       0.266017   42  
                      1     0.814286  0.828194  0.890181       0.367784   42  
                      2     0.790476  0.796296  0.869372       0.265855   42  
                      3     0.800000  0.801887  0.869735       0.239069   42  
                      4     0.804762  0.809302  0.874092       0.302706   42  
csp_lda       inter   0     0.557143  0.593886  0.594786       0.009230   42  
                      1     0.576190  0.597285  0.626033       0.033666   42  
                      2     0.614286  0.612440  0.683733       0.108967   42  
                      3     0.609524  0.620370  0.660948       0.075664   42  
                      4     0.580952  0.617391  0.654503       0.082697   42  
csp_svm       inter   0     0.609524  0.702899  0.618948       0.057102   42  
                      1     0.580952  0.661538  0.680670       0.098942   42  
                      2     0.661905  0.714859  0.746097       0.176676   42  
                      3     0.704762  0.757812  0.776961       0.231083   42  
                      4     0.590476  0.671756  0.689815       0.094722   42  
eegnet        inter   0     0.595238  0.717608  0.815424      -0.525424   42  
                      1     0.519048  0.683386  0.6002

In [19]:
literature_agg = literature_df["test"].groupby(level=0).agg(["mean", "std"]).stack(0).swaplevel(0,1).sort_index()
literature_agg

mean       std
               model                            
accuracy       atcnet         0.607619  0.053938
               csp_lda        0.587619  0.023952
               csp_svm        0.629524  0.052424
               eeg_conformer  0.497143  0.018007
               eegnet         0.603810  0.055164
               siglr          0.800000  0.010102
d2_brier_score atcnet        -0.425346  0.184724
               csp_lda        0.062045  0.040026
               csp_svm        0.131705  0.070543
               eeg_conformer -0.954590  0.068347
               eegnet        -0.481662  0.292169
               siglr          0.288286  0.049873
f1             atcnet         0.637045  0.199227
               csp_lda        0.608275  0.011985
               csp_svm        0.701773  0.038186
               eeg_conformer  0.272526  0.373175
               eegnet         0.698053  0.029304
               siglr          0.805626  0.014119
roc_auc        atcnet         0.772214  0.030900
               csp_lda        0.644000  0.034355
               csp_svm        0.702498  0.061367
               eeg_conformer  0.571546  0.076907
               eegnet         0.732633  0.081511
               siglr          0.875642  0.008495

## Sig2+$\ell_1$

In [ ]:
DEGREE = 2

literature_models = {
    "siglr": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, random_state=SEED, l1_ratio=1.0)
}
literature_df = evaluate_models_with_subject_dict(literature_models, subject_dict, random_state=SEED)
literature_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   57.1s finished


train                                         test  \
                    accuracy        f1   roc_auc d2_brier_score  accuracy   
model subject fold                                                          
siglr inter   0     0.930952  0.933025  0.979374       0.731829  0.823810   
              1     0.927381  0.930126  0.979748       0.735810  0.785714   
              2     0.942857  0.944954  0.984904       0.752048  0.809524   
              3     0.940476  0.942396  0.980908       0.736561  0.804762   
              4     0.938095  0.940230  0.981675       0.737583  0.790476   

                                                      seed  
                          f1   roc_auc d2_brier_score       
model subject fold                                          
siglr inter   0     0.834081  0.883005       0.456674   42  
              1     0.800000  0.870379       0.421743   42  
              2     0.818182  0.865832       0.409844   42  
              3     0.805687  0.887981       0.450479   42  
              4     0.798165  0.883442       0.434314   42

In [ ]:
literature_agg = literature_df["test"].groupby(level=0).agg(["mean", "std"]).stack(0).swaplevel(0,1).sort_index()
literature_agg

,,mean,std
,model,,
accuracy,siglr,0.802857,0.015283
d2_brier_score,siglr,0.434611,0.019478
f1,siglr,0.811223,0.014983
roc_auc,siglr,0.878128,0.009492


## 3rd order of signature

In [ ]:
DEGREE=3
literature_models = {
    "siglr": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, random_state=SEED),
    #"siglr": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, random_state=SEED, l1_ratio=1.0)
}
literature_df = evaluate_models_with_subject_dict(literature_models, subject_dict, random_state=SEED)
literature_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed: 12.9min finished


train                                  test            \
                   accuracy   f1 roc_auc d2_brier_score  accuracy        f1   
model subject fold                                                            
siglr inter   0         1.0  1.0     1.0            1.0  0.709524  0.708134   
              1         1.0  1.0     1.0            1.0  0.742857  0.752294   
              2         1.0  1.0     1.0            1.0  0.690476  0.700461   
              3         1.0  1.0     1.0            1.0  0.766667  0.782222   
              4         1.0  1.0     1.0            1.0  0.728571  0.732394   

                                            seed  
                     roc_auc d2_brier_score       
model subject fold                                
siglr inter   0     0.765828      -0.111897   42  
              1     0.770370      -0.014479   42  
              2     0.746369      -0.182322   42  
              3     0.817130       0.128246   42  
              4     0.797613      -0.024961   42

In [ ]:
literature_agg = literature_df["test"].groupby(level=0).agg(["mean", "std"]).stack(0).swaplevel(0,1).sort_index()
literature_agg

,,mean,std
,model,,
accuracy,siglr,0.727619,0.029431
d2_brier_score,siglr,-0.041083,0.116836
f1,siglr,0.735101,0.033356
roc_auc,siglr,0.779462,0.027893


In [ ]:
DEGREE=3
literature_models = {
    #"siglr": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, random_state=SEED),
    "siglr": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, random_state=SEED, l1_ratio=1.0)
}
literature_df = evaluate_models_with_subject_dict(literature_models, subject_dict, random_state=SEED)
literature_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed: 55.9min finished


train                                         test  \
                    accuracy        f1   roc_auc d2_brier_score  accuracy   
model subject fold                                                          
siglr inter   0     0.986905  0.987400  0.999694       0.894212  0.723810   
              1     0.986905  0.987400  0.999444       0.887440  0.752381   
              2     0.985714  0.986301  0.999398       0.890581  0.704762   
              3     0.984524  0.985075  0.999376       0.886015  0.757143   
              4     0.986905  0.987457  0.999432       0.887048  0.742857   

                                                      seed  
                          f1   roc_auc d2_brier_score       
model subject fold                                          
siglr inter   0     0.731481  0.794350       0.233591   42  
              1     0.769912  0.809701       0.273715   42  
              2     0.718182  0.754902       0.134215   42  
              3     0.773333  0.833878       0.344229   42  
              4     0.761062  0.810004       0.277289   42

In [ ]:
literature_agg = literature_df["test"].groupby(level=0).agg(["mean", "std"]).stack(0).swaplevel(0,1).sort_index()
literature_agg

,,mean,std
,model,,
accuracy,siglr,0.736190,0.021718
d2_brier_score,siglr,0.252608,0.077181
f1,siglr,0.750794,0.024574
roc_auc,siglr,0.800567,0.029181


#Hyperparameter study for inter

In [ ]:
DEGREE = 2
aug_models = {
    "siglr_no_aug": SigLR(DEGREE, time_aug=False, lead_lag_aug=False, random_state=SEED),
    "siglr_time_aug": SigLR(DEGREE, time_aug=True, lead_lag_aug=False, random_state=SEED),
    "siglr_lead_lag_aug": SigLR(DEGREE, time_aug=False, lead_lag_aug=True, random_state=SEED),
    "siglr_both_aug": SigLR(DEGREE, time_aug=True, lead_lag_aug=True, random_state=SEED),
}

In [ ]:
aug_df = evaluate_models_with_subject_dict(aug_models, subject_dict, random_state=SEED)
aug_df

[Parallel(n_jobs=5)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=5)]: Done   2 out of   5 | elapsed:    4.5s remaining:    6.7s
[Parallel(n_jobs=5)]: Done   5 out of   5 | elapsed:    4.7s finished
[Parallel(n_jobs=5)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=5)]: Done   2 out of   5 | elapsed:    2.6s remaining:    3.9s
[Parallel(n_jobs=5)]: Done   5 out of   5 | elapsed:    2.8s finished
[Parallel(n_jobs=5)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=5)]: Done   2 out of   5 | elapsed:   12.2s remaining:   18.3s
[Parallel(n_jobs=5)]: Done   5 out of   5 | elapsed:   12.6s finished
[Parallel(n_jobs=5)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=5)]: Done   2 out of   5 | elapsed:   12.3s remaining:   18.5s
[Parallel(n_jobs=5)]: Done   5 out of   5 | elapsed:   12.5s finished


train                              \
                                accuracy   f1 roc_auc d2_brier_score   
model              subject fold                                        
siglr_no_aug       inter   0         1.0  1.0     1.0       0.999194   
                           1         1.0  1.0     1.0       0.999349   
                           2         1.0  1.0     1.0       0.999374   
                           3         1.0  1.0     1.0       0.999488   
                           4         1.0  1.0     1.0       0.999310   
siglr_time_aug     inter   0         1.0  1.0     1.0       0.999395   
                           1         1.0  1.0     1.0       0.999243   
                           2         1.0  1.0     1.0       0.999483   
                           3         1.0  1.0     1.0       0.999488   
                           4         1.0  1.0     1.0       0.999593   
siglr_lead_lag_aug inter   0         1.0  1.0     1.0       0.999950   
                           1         1.0  1.0     1.0       0.999983   
                           2         1.0  1.0     1.0       0.999981   
                           3         1.0  1.0     1.0       0.999962   
                           4         1.0  1.0     1.0       0.999957   
siglr_both_aug     inter   0         1.0  1.0     1.0       0.999946   
                           1         1.0  1.0     1.0       0.999979   
                           2         1.0  1.0     1.0       0.999973   
                           3         1.0  1.0     1.0       0.999970   
                           4         1.0  1.0     1.0       0.999979   

                                     test                                     \
                                 accuracy        f1   roc_auc d2_brier_score   
model              subject fold                                                
siglr_no_aug       inter   0     0.757143  0.767123  0.829594       0.127534   
                           1     0.776190  0.791111  0.847125       0.258135   
                           2     0.733333  0.733333  0.788671       0.057699   
                           3     0.709524  0.705314  0.765704      -0.040664   
                           4     0.719048  0.712195  0.811093       0.040934   
siglr_time_aug     inter   0     0.757143  0.762791  0.843492       0.183202   
                           1     0.780952  0.792793  0.855391       0.288136   
                           2     0.747619  0.751174  0.800654       0.083890   
                           3     0.723810  0.721154  0.778322      -0.007656   
                           4     0.719048  0.717703  0.825163       0.071701   
siglr_lead_lag_aug inter   0     0.790476  0.790476  0.863748       0.252889   
                           1     0.795238  0.807175  0.879826       0.294864   
                           2     0.780952  0.790909  0.860657       0.205160   
                           3     0.790476  0.792453  0.852669       0.205892   
                           4     0.790476  0.794393  0.870280       0.251414   
siglr_both_aug     inter   0     0.800000  0.801887  0.867563       0.277736   
                           1     0.809524  0.819820  0.881733       0.319488   
                           2     0.780952  0.790909  0.864651       0.216665   
                           3     0.785714  0.786730  0.858932       0.228586   
                           4     0.780952  0.787037  0.867102       0.223789   

                                seed  
                                      
model              subject fold       
siglr_no_aug       inter   0      42  
                           1      42  
                           2      42  
                           3      42  
                           4      42  
siglr_time_aug     inter   0      42  
                           1      42  
                           2      42  
                           3      42  
                           4      42  
siglr_lead_lag_aug inter   0  

In [ ]:
aug_agg = aug_df["test"].groupby(level=0).agg(["mean", "std"]).stack(0).swaplevel(0,1).sort_index()
aug_agg

mean       std
               model                                 
accuracy       siglr_both_aug      0.791429  0.012778
               siglr_lead_lag_aug  0.789524  0.005216
               siglr_no_aug        0.739048  0.027438
               siglr_time_aug      0.745714  0.025332
d2_brier_score siglr_both_aug      0.253253  0.044164
               siglr_lead_lag_aug  0.242044  0.037626
               siglr_no_aug        0.088728  0.112020
               siglr_time_aug      0.123855  0.114152
f1             siglr_both_aug      0.797277  0.014019
               siglr_lead_lag_aug  0.795081  0.006933
               siglr_no_aug        0.741815  0.036579
               siglr_time_aug      0.749123  0.031095
roc_auc        siglr_both_aug      0.867996  0.008412
               siglr_lead_lag_aug  0.865436  0.010237
               siglr_no_aug        0.808437  0.032281
               siglr_time_aug      0.820604  0.031387